# Case study 1: structural coverage of a cancer gene panel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaelmdc/braidworks/blob/main/notebooks/case-studies/cancer-gene-structures.ipynb)

**Question.** For 20 frequently mutated human cancer genes, how many experimental
structures exist, how confident is the AlphaFold prediction, and how many Reactome
pathways does each protein take part in?

**Route.** Gene symbols → UniProt (accession and name) → PDBe (experimental
structures), AlphaFold DB (mean pLDDT) and Reactome (pathways), planned automatically
from one request. Every value is retrieved live; nothing is downloaded in bulk.

## Setup

Install Braidworks with every public source, then record the environment.

In [ ]:
%pip install -q "braidworks[pandas]" matplotlib

In [ ]:
import datetime
import importlib.metadata
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt

import braidworks as bw

OUT = Path("case-study-output")
OUT.mkdir(exist_ok=True)

# Figure style: one hue, muted axes, hairline grid (print-friendly, colour-blind safe).
INK, MUTED, GRID, ACCENT = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6"
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False,
})


def record_environment(sources):
    """Versions and date, saved next to the outputs so a run can be reproduced."""
    packages = ["braidworks-core", *sources]
    versions = {name: importlib.metadata.version(name) for name in packages}
    env = {
        "date_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        **versions,
    }
    for key, value in env.items():
        print(f"{key:24} {value}")
    return env


def timed(label, run):
    start = time.perf_counter()
    result = run()
    seconds = time.perf_counter() - start
    print(f"{label}: {seconds:.2f} s  ({result.summary()})")
    return result, seconds

In [ ]:
env = record_environment(['uniprot_weaver', 'pdbe_weaver', 'alphafold_weaver', 'reactome_weaver'])

## The request

The panel is ordinary gene symbols; `organism="9606"` restricts them to human.

In [ ]:
GENES = [
    "TP53", "BRCA1", "BRCA2", "EGFR", "KRAS", "PTEN", "MYC", "PIK3CA", "APC", "RB1",
    "CDKN2A", "ATM", "ERBB2", "BRAF", "NRAS", "IDH1", "VHL", "SMAD4", "CTNNB1", "NOTCH1",
]
WANT = [
    "protein.name",
    "structure.pdb.ids",
    "structure.alphafold.mean_plddt",
    "pathway.reactome.names",
]
print(bw.path("protein.query", WANT))

## Run it, cold and warm

The first run starts from an empty lookup cache, so every source is called. The second
reuses the cache, as a rerun of the analysis would.

In [ ]:
cache = OUT / "cancer-gene-structures-cache.sqlite"
cache.unlink(missing_ok=True)
with bw.Session(cache=cache) as session:
    result, cold_s = timed("cold", lambda: session.get(GENES, have="protein.query", want=WANT, organism="9606"))
with bw.Session(cache=cache) as session:
    _, warm_s = timed("warm", lambda: session.get(GENES, have="protein.query", want=WANT, organism="9606"))

In [ ]:
table = result.to_pandas()[[
    "input", "status", "protein.name", "structure.pdb.ids.total",
    "structure.alphafold.mean_plddt", "pathway.reactome.names.total",
]].rename(columns={
    "input": "gene", "protein.name": "protein", "structure.pdb.ids.total": "pdb_structures",
    "structure.alphafold.mean_plddt": "alphafold_mean_plddt",
    "pathway.reactome.names.total": "reactome_pathways",
})
table.to_csv(OUT / "cancer-gene-structures.csv", index=False)
result.to_csv(OUT / "cancer-gene-structures-full.csv")
table

Proteins longer than AlphaFold DB's single-model limit (about 2,700 residues, such as
BRCA2, ATM and APC) have no single predicted model, so their pLDDT is empty rather than
estimated. The `missing` column of the full CSV names each gap.

## Figure

In [ ]:
plotted = table.dropna(subset=["alphafold_mean_plddt"])
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(plotted["pdb_structures"], plotted["alphafold_mean_plddt"], s=28, color=ACCENT,
           edgecolor="white", linewidth=1.2, zorder=3)
points = list(zip(plotted["gene"], plotted["pdb_structures"], plotted["alphafold_mean_plddt"]))
for gene, x, y in points:
    # Label to the left when a close neighbour sits just to the right, so labels don't collide.
    crowded = any(x < x2 <= x * 1.5 and abs(y - y2) < 4 for _, x2, y2 in points)
    ax.annotate(gene, (x, y), xytext=(-4 if crowded else 4, 2), textcoords="offset points",
                ha="right" if crowded else "left", fontsize=7, color=INK)
ax.set_xscale("log")
ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000)
         if plotted["pdb_structures"].min() / 2 <= t <= plotted["pdb_structures"].max() * 2]
ax.set_xticks(ticks, [str(t) for t in ticks])
ax.minorticks_off()
ax.set_xlabel("Experimental PDB structures (log scale)")
ax.set_ylabel("AlphaFold mean pLDDT")
ax.set_ylim(top=100)
gaps = ", ".join(sorted(set(table["gene"]) - set(plotted["gene"])))
ax.set_title(f"{len(plotted)} of {len(table)} genes (no single AlphaFold model: {gaps})",
             fontsize=8, color=MUTED, loc="left")
fig.tight_layout()
fig.savefig(OUT / "cancer-gene-structures.png")
fig.savefig(OUT / "cancer-gene-structures.svg")

In [ ]:
import json

env["timings_s"] = {"cold": round(cold_s, 2), "warm": round(warm_s, 2)}
(OUT / "cancer-gene-structures-environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env["timings_s"]))

## Sources to cite

In [ ]:
print(result.cite())

## Reproducing this case study

The sources are live databases, so counts change as they are updated: compare your
numbers with the environment record and date saved in `cancer-gene-structures-environment.json`, not
with the figures in a paper. Timings depend on the network and on provider load; the
warm run shows what a lookup cache saves when an analysis is rerun.